<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/generation-4_Qwen3.5-2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVLM Dashboard Insight Generation

## Initial Steps

In [1]:
!pip install -q supabase pillow requests torch torchvision einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.3 MB/s eta 0:00:00


In [2]:
import os
import time
import requests
import torch
from io import BytesIO
from PIL import Image
from supabase import create_client, Client
from google.colab import userdata
from huggingface_hub import login

In [3]:
# Supabase credentials
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client initialised.")

login(token=HF_TOKEN)
print("HuggingFace login successful.")

Supabase client initialised.
HuggingFace login successful.


In [4]:
# Pull qualifying metadata_ids from human_insights
hi_response = supabase.table("human_insights") \
    .select("metadata_id") \
    .eq("expected_dataset", True) \
    .is_("rejection_reason", "null") \
    .execute()

qualified_ids = list({row["metadata_id"] for row in hi_response.data})
print(f"Qualified dashboards: {len(qualified_ids)}")

# Pull metadata only for those ids
response = supabase.table("metadata") \
    .select("id, bucket_path") \
    .in_("id", qualified_ids) \
    .execute()
dashboards = response.data

print(f"Loaded {len(dashboards)} dashboards.")
print("Sample record:", dashboards[0] if dashboards else "(empty)")

Qualified dashboards: 40
Loaded 40 dashboards.
Sample record: {'id': 'abee2e83-6384-4c23-abfd-e5ede8b5a7bf', 'bucket_path': 'screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png'}


In [5]:
# Build public image URL from bucket_path
def build_image_url(bucket_path: str) -> str:
    return f"{SUPABASE_URL}/storage/v1/object/public/superstore/{bucket_path}"

# Fetch image from URL and return a PIL Image
def fetch_image(url: str) -> Image.Image:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")

# Resize the image prior the inference pipeline
def prepare_image(image: Image.Image, max_width=2000, max_height=1500) -> Image.Image:
    if image.width > max_width or image.height > max_height:
        ratio = min(max_width / image.width, max_height / image.height)
        new_size = (int(image.width * ratio), int(image.height * ratio))
        image = image.resize(new_size)
        print(f"  Resized to {new_size}")
    return image

# Extract model identity from a loaded model object
def get_model_meta(model, hf_id=None):
    cfg   = getattr(model, "config", None)
    hf_id = hf_id or getattr(cfg, "_name_or_path", None)
    name  = hf_id.split("/")[-1] if hf_id else None
    return {"model_name": name, "model_hf_id": hf_id}

In [6]:
# Prompt
PROMPT = """
You are a senior BI analyst generating structured insights for a business intelligence dashboard.
Think step-by-step: first extract visible quantitative facts, then identify visual patterns, then derive business implications.

Before writing any chart analysis, count the number of distinct charts visible in the dashboard and write: 'Chart count: N'.
Then produce exactly N chart analyses and no more.
Analyze chart-by-chart in Z-pattern (left to right, top to bottom).
If there are scoreboard/scorecard charts (e.g., sales, profit, orders, customers, etc), treat them as the first chart as one single chart with the title of "Scoreboard Overview".
Once grouped into Scoreboard Overview, those KPI panels are fully analyzed and must never appear again as individual charts anywhere in your output.
If there are no scoreboard/scorecard charts, proceed with writing the first chart available.
Do not treat UI labels, navigation tabs, filters, or sidebar controls as charts.
For each chart, write exactly:
L2: one sentence reporting only values explicitly shown or labeled: highest/lowest value, comparison, ranking, or proportion only. Do not compute anything not displayed in the image.
L3: one sentence describing a visual pattern: a direction, a shape, a gap, or an exception. Use natural language: "volatile", "dipped", "wider margin", "considerably far", "spread". Use hedging: "appears to", "seems to", "suggesting". Write NOT APPLICABLE if the chart is: a ranked table, a top-N list, or a gauge.
L4: one sentence connecting the pattern to business context or domain knowledge not visible in the chart. Must reference a specific value from L2 or a specific pattern from L3; never use generic phrases such as 'this could be due to' without grounding them in what was observed. Never restate what is already visible. Always required.

Output format:
Chart 1: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Chart 2: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Rules:
- Write EXACTLY 1 sentence per level per chart
- Skip navigation tabs, filters, sidebar controls, and dropdowns entierly
- Do not include axis labels, colors, or chart type names
- Immediately after writing your final chart analysis, write END OF ANALYSIS on its own line and generate no further text under any circumstances
"""

In [7]:
# Quick sanity check on the first dashboard
if dashboards:
    sample_url = build_image_url(dashboards[0]["bucket_path"])
    print("Sample URL:", sample_url)
    sample_img = fetch_image(sample_url)
    print("Image size:", sample_img.size)
    sample_img

Sample URL: https://olduvnqhykovcfbfouhe.supabase.co/storage/v1/object/public/superstore/screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png
Image size: (1200, 927)


---

## Qwen3.5-2B
https://huggingface.co/Qwen/Qwen3.5-2B  

In [8]:
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"

  Cloning https://github.com/huggingface/transformers.git (to revision main) to /tmp/pip-install-a282uizc/transformers_4a835349925d4d498fa852469b6e5184
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-install-a282uizc/transformers_4a835349925d4d498fa852469b6e5184
  Resolved https://github.com/huggingface/transformers.git to commit eed95d8c445b8679ba342cffa947a3ed2b8d7fbc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11655128 sha256=a7bc30d14f414363864036a2b97178a13d1ae2d38b7133c91e2cbe4d0a794f15
  Stored in directory: /tmp/pip-ephem-wheel-cache-2nkqmvig/wheels/12/51/df/b62c8ce0479c5de6f7bef121169b3e946949a57481169d3155
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninsta

In [9]:
!pip install -q qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 24.8 MB/s eta 0:00:00


In [10]:
import transformers
print(transformers.__version__)
print([x for x in dir(transformers) if 'Qwen3' in x])

5.8.0.dev0
['Qwen3Config', 'Qwen3ForCausalLM', 'Qwen3ForQuestionAnswering', 'Qwen3ForSequenceClassification', 'Qwen3ForTokenClassification', 'Qwen3Model', 'Qwen3MoeConfig', 'Qwen3MoeForCausalLM', 'Qwen3MoeForQuestionAnswering', 'Qwen3MoeForSequenceClassification', 'Qwen3MoeForTokenClassification', 'Qwen3MoeModel', 'Qwen3MoePreTrainedModel', 'Qwen3NextConfig', 'Qwen3NextForCausalLM', 'Qwen3NextForQuestionAnswering', 'Qwen3NextForSequenceClassification', 'Qwen3NextForTokenClassification', 'Qwen3NextModel', 'Qwen3NextPreTrainedModel', 'Qwen3OmniMoeAudioEncoderConfig', 'Qwen3OmniMoeCode2Wav', 'Qwen3OmniMoeCode2WavDecoderBlock', 'Qwen3OmniMoeCode2WavTransformerModel', 'Qwen3OmniMoeConfig', 'Qwen3OmniMoeForConditionalGeneration', 'Qwen3OmniMoePreTrainedModel', 'Qwen3OmniMoePreTrainedModelForConditionalGeneration', 'Qwen3OmniMoeProcessor', 'Qwen3OmniMoeTalkerCodePredictorConfig', 'Qwen3OmniMoeTalkerCodePredictorModel', 'Qwen3OmniMoeTalkerCodePredictorModelForConditionalGeneration', 'Qwen3Omni

In [11]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
print("transformers:", transformers.__version__)

transformers: 5.8.0.dev0


In [12]:
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor

MODEL_HF_ID = "Qwen/Qwen3.5-2B"
device      = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    MODEL_HF_ID,
    token=HF_TOKEN,
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_HF_ID,
    torch_dtype=torch.bfloat16,
    device_map=device,
    token=HF_TOKEN,
).eval()

meta = get_model_meta(model, hf_id=MODEL_HF_ID)
print(f"Loaded on {device}")
print(type(model))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Loaded on cuda
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>


### Testing one sample generation

In [13]:
# from qwen_vl_utils import process_vision_info

# test_dashboard = dashboards[0]
# test_image     = fetch_image(build_image_url(test_dashboard["bucket_path"]))

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image", "image": test_image},
#             {"type": "text",  "text": PROMPT},
#         ],
#     }
# ]

# text = processor.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
# )
# image_inputs, video_inputs = process_vision_info(messages)
# inputs = processor(
#     text=[text],
#     images=image_inputs,
#     videos=video_inputs,
#     return_tensors="pt",
# ).to(device)

# t0 = time.perf_counter()
# with torch.no_grad():
#     generated_ids = model.generate(
#         **inputs,
#         max_new_tokens=1024,
#         do_sample=False,
#     )
# test_output = processor.batch_decode(
#     generated_ids[:, inputs["input_ids"].shape[1]:],
#     skip_special_tokens=True,
# )[0]
# test_ms = int((time.perf_counter() - t0) * 1000)

# print(f"Dashboard ID:   {test_dashboard['id']}")
# print(f"Inference time: {test_ms} ms")
# print(f"\nOutput:\n{test_output}")
# display(test_image)

# supabase.table("vlm_outputs").upsert({
#     "metadata_id":       test_dashboard["id"],
#     **meta,
#     "raw_output":        test_output,
#     "inference_success": True,
#     "error_message":     None,
#     "inference_ms":      test_ms,
# }, on_conflict="metadata_id,model_name").execute()

# print("Saved to vlm_outputs.")

### 40 dashboards generation

In [14]:
from qwen_vl_utils import process_vision_info
from tqdm import tqdm

ok  = 0
err = 0

for dashboard in tqdm(dashboards, desc="Generating", unit="dashboard"):
    dashboard_id = dashboard["id"]

    try:
        image = prepare_image(fetch_image(build_image_url(dashboard["bucket_path"])))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": PROMPT},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(device)

        t0 = time.perf_counter()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        elapsed_ms = int((time.perf_counter() - t0) * 1000)

        supabase.table("vlm_outputs").upsert({
            "metadata_id":       dashboard_id,
            **meta,
            "raw_output":        output,
            "inference_success": True,
            "error_message":     None,
            "inference_ms":      elapsed_ms,
        }, on_conflict="metadata_id,model_name").execute()

        ok += 1
        print(f"[OK]  {dashboard_id}  ({elapsed_ms} ms)")
        print(f"      {output[:120]}...\n")

    except Exception as e:
        supabase.table("vlm_outputs").upsert({
            "metadata_id":       dashboard_id,
            **meta,
            "raw_output":        None,
            "inference_success": False,
            "error_message":     str(e),
            "inference_ms":      None,
        }, on_conflict="metadata_id,model_name").execute()

        err += 1
        print(f"[ERR] {dashboard_id}: {e}")

print(f"\nDone. {ok} succeeded, {err} failed.")

Generating:   2%|▎         | 1/40 [01:07<43:58, 67.66s/dashboard]

[OK]  abee2e83-6384-4c23-abfd-e5ede8b5a7bf  (66971 ms)
      Chart 1: Scoreboard Overview
L2: Highest value is $733,215 (Current Year Sales), lowest is $146.4K (California Sales).
L...



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Generating:   5%|▌         | 2/40 [02:36<50:44, 80.12s/dashboard]

[OK]  ee87f028-0bf2-4c03-81c5-6b974a4cfcb5  (88010 ms)
      Chart 1: Scoreboard Overview
L2: Sales: $60.7K, Profit: $15.0K, Total Orders: 245, Total Customers: 118
L3: Sales shows ...



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Generating:   8%|▊         | 3/40 [03:49<47:21, 76.78s/dashboard]

...

[OK]  1c6fba5d-67a4-4f86-b7a8-6f5f37c4dd30  (58078 ms)
      Chart 1: Scoreboard Overview
L2: Sales peaked at $84K, Quantity peaked at 1,723, and Profit peaked at $93K.
L3: The Sale...



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Generating:  98%|█████████▊| 39/40 [52:14<01:03, 63.44s/dashboard]

[OK]  aa95ffcc-e6f4-4869-9784-92bd011b3b79  (42975 ms)
      Chart 1: Monthly Sales Performance by Category
L2: Chairs show the highest sales volume at 14,966 units, followed by Sto...



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Generating: 100%|██████████| 40/40 [53:37<00:00, 80.44s/dashboard]

[OK]  932c11c0-d78c-4651-8bb9-73325937333d  (82859 ms)
      Chart 1: Scoreboard Overview
L2: Sales ($733.22K) is the highest value, followed by Profit ($93.44K), Orders (1,687), an...


Done. 39 succeeded, 1 failed.


### Check failed dashboard and regenerate

In [15]:
response = supabase.table("vlm_outputs") \
    .select("metadata_id, error_message") \
    .eq("model_name", "Qwen3.5-2B") \
    .eq("inference_success", False) \
    .execute()

for row in response.data:
    print(f"Dashboard ID: {row['metadata_id']}")
    print(f"Error: {row['error_message']}")

Dashboard ID: 0680041e-4ba2-4935-8f7e-02f264285350
Error: CUDA out of memory. Tried to allocate 1.19 GiB. GPU 0 has a total capacity of 14.56 GiB of which 319.81 MiB is free. Including non-PyTorch memory, this process has 14.25 GiB memory in use. Of the allocated memory 14.01 GiB is allocated by PyTorch, and 118.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


In [16]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [17]:
dashboard = supabase.table("metadata") \
    .select("id, bucket_path") \
    .eq("id", "0680041e-4ba2-4935-8f7e-02f264285350") \
    .execute().data[0]

image = fetch_image(build_image_url(dashboard["bucket_path"]))
print(f"Image size: {image.size}")

Image size: (3200, 1854)


In [18]:
import torch
torch.cuda.empty_cache()

dashboard_id = "0680041e-4ba2-4935-8f7e-02f264285350"
image = fetch_image(build_image_url(dashboard["bucket_path"]))
image = image.resize((1600, 927))  # 50% downscale

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

t0 = time.perf_counter()
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]
elapsed_ms = int((time.perf_counter() - t0) * 1000)

supabase.table("vlm_outputs").upsert({
    "metadata_id":       dashboard_id,
    **meta,
    "raw_output":        output,
    "inference_success": True,
    "error_message":     None,
    "inference_ms":      elapsed_ms,
}, on_conflict="metadata_id,model_name").execute()

print(f"Done. ({elapsed_ms} ms)")
print(f"\nOutput:\n{output}")

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Done. (96103 ms)

Output:
Chart 1: Scoreboard Overview
L2: The highest value is $733,215 (SALES), followed by $93,439 (PROFIT), $1,687 (ORDERS), and 693 (CUSTOMERS).
L3: The Sales chart shows a significant spike in the final month (November), while the Profit chart shows a sharp dip in the same period.
L4: This divergence suggests a potential revenue surge in November that may have temporarily eroded profit margins, indicating a need to investigate the underlying cause of the sales increase.

Chart 2: Sales by State
L2: The highest value is 146,388, and the lowest value is 0.
L3: The map displays a clear geographic concentration of sales in the Northeast and Midwest, with a noticeable absence of data in the South and West.
L4: This spatial distribution suggests that the business is heavily concentrated in urban or suburban markets, potentially limiting growth opportunities in rural or less developed regions.

Chart 3: Sales by Segment
L2: The highest value is $331,905 (Consumer), follo